# Density Correlation Functions

This notebook demonstrates density correlation analysis in pyscal3:

- **Structure factor S(k)**: Fourier transform of density correlations, comparable to scattering experiments
- **Local density**: Per-atom density using Voronoi, neighbor-count, or Gaussian methods
- **Density fluctuations**: Statistical analysis via block decomposition
- **Hyperuniformity**: Classification of long-range density correlations

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ase.build import bulk
import pyscal3

## Structure Factor S(k)

The static structure factor is defined as:

$$S(k) = \frac{1}{N} \left| \sum_j e^{i\mathbf{k} \cdot \mathbf{r}_j} \right|^2$$

It's directly related to X-ray and neutron scattering intensity. Key physical interpretations:
- S(k→0) is proportional to isothermal compressibility
- Peak positions indicate interatomic spacings
- Peak heights measure short-range order

In [ ]:
# Create FCC copper crystal
atoms_fcc = bulk("Cu", "fcc", cubic=True).repeat(4)

# Compute structure factor
result = pyscal3.structure_factor(atoms_fcc, k_max=15.0, n_k=100, n_samples=100)

print(f"Number of atoms: {len(atoms_fcc)}")
print(f"S(k→0) ≈ {result['S_0']:.3f}")

In [ ]:
# Plot S(k)
plt.figure(figsize=(10, 5))
plt.plot(result['k'], result['S'], 'b-', lw=1.5)
plt.xlabel('k (Å⁻¹)', fontsize=12)
plt.ylabel('S(k)', fontsize=12)
plt.title('Static Structure Factor for FCC Copper')
plt.grid(True, alpha=0.3)
plt.xlim(0, 15)
plt.show()

### Comparing Different Structures

Let's compare S(k) for FCC, BCC, and a slightly disordered structure.

In [ ]:
# Create different structures
atoms_bcc = bulk("Fe", "bcc", cubic=True).repeat(4)

# Disordered structure (randomized positions)
atoms_disordered = bulk("Cu", "fcc", cubic=True).repeat(4)
pos = atoms_disordered.get_positions()
np.random.seed(42)
pos += np.random.randn(*pos.shape) * 0.3  # Add noise
atoms_disordered.set_positions(pos)

# Compute S(k) for each
S_fcc = pyscal3.structure_factor(atoms_fcc, k_max=12.0, n_k=80, n_samples=80)
S_bcc = pyscal3.structure_factor(atoms_bcc, k_max=12.0, n_k=80, n_samples=80)
S_dis = pyscal3.structure_factor(atoms_disordered, k_max=12.0, n_k=80, n_samples=80)

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(S_fcc['k'], S_fcc['S'], 'b-', label='FCC Cu', lw=1.5)
plt.plot(S_bcc['k'], S_bcc['S'], 'r-', label='BCC Fe', lw=1.5)
plt.plot(S_dis['k'], S_dis['S'], 'g-', label='Disordered', lw=1.5, alpha=0.7)
plt.xlabel('k (Å⁻¹)', fontsize=12)
plt.ylabel('S(k)', fontsize=12)
plt.title('Structure Factor Comparison')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## Local Density

Local density measures how crowded each atom's environment is. Three methods are available:

1. **Voronoi**: ρᵢ = 1/Vᵢ (inverse Voronoi cell volume)
2. **Neighbor**: ρᵢ = nᵢ / (4πr³/3) (neighbors in sphere)
3. **Gaussian**: Smoothed density field

In [ ]:
# Voronoi-based local density
atoms = bulk("Cu", "fcc", cubic=True).repeat(4)
pyscal3.find_neighbors(atoms, method="voronoi")

rho_voronoi = pyscal3.local_density(atoms, method='voronoi')

print(f"Voronoi local density:")
print(f"  Mean: {rho_voronoi['mean']:.4f} atoms/Å³")
print(f"  Std:  {rho_voronoi['std']:.6f}")
print(f"  CV:   {rho_voronoi['std']/rho_voronoi['mean']*100:.3f}%")  # Coefficient of variation

In [ ]:
# Compare methods on disordered structure
atoms_dis = atoms_disordered.copy()
pyscal3.find_neighbors(atoms_dis, method="voronoi")

rho_v = pyscal3.local_density(atoms_dis, method='voronoi')
rho_n = pyscal3.local_density(atoms_dis, method='neighbor', cutoff=3.0)
rho_g = pyscal3.local_density(atoms_dis, method='gaussian', sigma=1.0)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))

for ax, rho, title in zip(axes, 
                          [rho_v, rho_n, rho_g], 
                          ['Voronoi', 'Neighbor (r=3Å)', 'Gaussian (σ=1Å)']):
    ax.hist(rho['density'], bins=30, color='steelblue', edgecolor='white')
    ax.axvline(rho['mean'], color='red', linestyle='--', label=f"Mean: {rho['mean']:.3f}")
    ax.set_xlabel('Local Density')
    ax.set_ylabel('Count')
    ax.set_title(title)
    ax.legend()

plt.tight_layout()
plt.show()

## Density Fluctuations

By dividing the simulation box into subcells, we can analyze density fluctuations:

$$\frac{\langle (\Delta N)^2 \rangle}{\langle N \rangle} = S(k \to 0) = \rho k_B T \kappa_T$$

This normalized variance is related to the isothermal compressibility κ_T.

In [ ]:
# Density fluctuations in a crystal
atoms = bulk("Cu", "fcc", cubic=True).repeat(5)
result = pyscal3.density_fluctuations(atoms, n_blocks=5)

print(f"Crystal density fluctuations:")
print(f"  Mean atoms per block: {result['mean_N']:.2f}")
print(f"  Variance: {result['var_N']:.4f}")
print(f"  Normalized variance: {result['normalized_variance']:.4f}")

In [ ]:
# Compare crystal vs. disordered
atoms_crystal = bulk("Cu", "fcc", cubic=True).repeat(5)
atoms_random = atoms_crystal.copy()
np.random.seed(123)
pos = atoms_random.get_positions()
pos += np.random.randn(*pos.shape) * 0.5
atoms_random.set_positions(pos)

fluct_crystal = pyscal3.density_fluctuations(atoms_crystal, n_blocks=5)
fluct_random = pyscal3.density_fluctuations(atoms_random, n_blocks=5)

print(f"Comparison:")
print(f"  Crystal normalized var: {fluct_crystal['normalized_variance']:.4f}")
print(f"  Random normalized var:  {fluct_random['normalized_variance']:.4f}")

In [ ]:
# Visualize block counts
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].hist(fluct_crystal['block_counts'], bins=15, color='steelblue', edgecolor='white')
axes[0].set_title('Crystal')
axes[0].set_xlabel('Atoms per block')
axes[0].set_ylabel('Count')

axes[1].hist(fluct_random['block_counts'], bins=15, color='coral', edgecolor='white')
axes[1].set_title('Disordered')
axes[1].set_xlabel('Atoms per block')

plt.tight_layout()
plt.show()

## Hyperuniformity Analysis

A system is **hyperuniform** if S(k) → 0 as k → 0, meaning large-scale density fluctuations are suppressed. This includes:
- All crystals (most strongly hyperuniform)
- Some disordered systems (e.g., jammed packings)

Hyperuniform systems are classified by the scaling S(k) ~ k^α:
- Class I: α > 1 (strongest, includes crystals)
- Class II: α = 1
- Class III: 0 < α < 1

In [ ]:
# Analyze hyperuniformity
atoms = bulk("Cu", "fcc", cubic=True).repeat(5)
result = pyscal3.hyperuniformity(atoms, k_max=6.0, n_k=60)

print(f"Hyperuniformity analysis:")
print(f"  Is hyperuniform: {result['is_hyperuniform']}")
print(f"  Class: {result['hyperuniform_class']}")
print(f"  Scaling exponent α: {result['alpha']:.3f}")
print(f"  Prefactor A: {result['A']:.3f}")

In [ ]:
# Plot S(k) from hyperuniformity analysis
S_k = result['S_k']

plt.figure(figsize=(10, 5))
plt.loglog(S_k['k'], S_k['S'], 'b.-', label='S(k)')

# Plot power law fit
k_fit = S_k['k'][S_k['k'] < 1.0]
S_fit = result['A'] * k_fit**result['alpha']
plt.loglog(k_fit, S_fit, 'r--', lw=2, label=f'Fit: S ~ k^{result["alpha"]:.2f}')

plt.xlabel('k (Å⁻¹)', fontsize=12)
plt.ylabel('S(k)', fontsize=12)
plt.title('Log-Log Structure Factor')
plt.legend()
plt.grid(True, alpha=0.3, which='both')
plt.show()

## Complete Workflow

Here's a complete analysis combining all density correlation functions.

In [ ]:
def analyze_density_correlations(atoms, name="Structure"):
    """Complete density correlation analysis."""
    print(f"\n=== {name} ({len(atoms)} atoms) ===")
    
    # Structure factor
    S = pyscal3.structure_factor(atoms, k_max=10.0, n_k=60, n_samples=60)
    print(f"S(k→0) ≈ {S['S_0']:.3f}")
    
    # Local density
    pyscal3.find_neighbors(atoms, method="voronoi")
    rho = pyscal3.local_density(atoms, method='voronoi')
    print(f"Local density: {rho['mean']:.4f} ± {rho['std']:.4f} atoms/Å³")
    
    # Density fluctuations
    fluct = pyscal3.density_fluctuations(atoms, n_blocks=4)
    print(f"Normalized variance: {fluct['normalized_variance']:.4f}")
    
    # Hyperuniformity
    hu = pyscal3.hyperuniformity(atoms, k_max=5.0, n_k=40)
    print(f"Hyperuniform: {hu['is_hyperuniform']} (α = {hu['alpha']:.2f})")
    
    return S, rho, fluct, hu

In [ ]:
# Analyze different structures
fcc = bulk("Cu", "fcc", cubic=True).repeat(4)
bcc = bulk("Fe", "bcc", cubic=True).repeat(4)
hcp = bulk("Mg", "hcp").repeat((4, 4, 4))

results = {}
for name, atoms in [('FCC Cu', fcc), ('BCC Fe', bcc), ('HCP Mg', hcp)]:
    results[name] = analyze_density_correlations(atoms, name)

## Summary

The density correlation functions in pyscal3 provide:

| Function | Purpose | Key Output |
|----------|---------|------------|
| `structure_factor()` | Fourier space correlations | S(k), S(0) |
| `local_density()` | Per-atom density | ρᵢ, mean, std |
| `density_fluctuations()` | Block statistics | ⟨ΔN²⟩/⟨N⟩ |
| `hyperuniformity()` | Long-range classification | α exponent, class |

These are particularly useful for:
- Comparing with scattering experiments
- Characterizing disorder in amorphous materials
- Studying phase transitions
- Identifying special ordered/hyperuniform states